In [6]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 데이터 로드
data = sns.load_dataset('titanic')

# 불필요한 컬럼 제거
data = data.drop(columns=['alive', 'deck', 'embark_town', 'class', 'who'], errors='ignore')

# 수치형, 범주형 변수 구분
num_features = ['age', 'fare', 'sibsp', 'parch']
cat_features = ['sex', 'embarked', 'alone']

# 결측치 처리 및 전처리 파이프라인
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # 수치형: 중앙값 대체
    ('scaler', StandardScaler())  # 표준화
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # 범주형: 최빈값 대체
    ('encoder', OneHotEncoder(handle_unknown='ignore'))  # 원-핫 인코딩 (drop_first 제거)
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

# 타겟 및 특징 변수 설정
X = data.drop(columns=['survived'], errors='ignore')
y = data['survived'].dropna()

# 결측치 제거 (타겟이 결측치인 경우)
X = X.loc[y.index]

# 로지스틱 회귀 모델 및 파이프라인
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear'))
])

# 교차 검증 수행
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

# 결과 출력
print(f'교차 검증 정확도 평균: {scores.mean():.4f}')
print(f'교차 검증 정확도 표준편차: {scores.std():.4f}')


교차 검증 정확도 평균: 0.7912
교차 검증 정확도 표준편차: 0.0168


### **Voting**
    - hard Voting
        - 모델 다수결 원칙, 예측 클래스 중 가장 많이 선택된 클래스 최종 예측값
    - Soft Voting
        - 예측 확률 -> 평균 내에서 최종 클래스 결정

### **Stacking**
    - 여러 개의 개별 모델을 기본 모델이랑 메타모델 식으로 해서 개별 모델로 독립적으로 예측 수행 -> 개별 모델들이 예측한 값을 입력 데이터로 사용
    - 최적의 조합으로 학습 -> 메타 모델이 예측 결과를 조합해서 최종 예측을 수행

In [11]:
## voting 방식

from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [17]:
from sklearn.ensemble import StackingClassifier

In [24]:
# 데이터 로드
data = sns.load_dataset('titanic')

# 불필요한 컬럼 제거
data = data.drop(columns=['alive', 'deck', 'embark_town', 'class', 'who'], errors='ignore')

# 수치형, 범주형 변수 구분
num_features = ['age', 'fare', 'sibsp', 'parch']
cat_features = ['sex', 'embarked', 'alone']

# 결측치 처리 및 전처리 파이프라인
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # 수치형: 중앙값 대체
    ('scaler', StandardScaler())  # 표준화
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # 범주형: 최빈값 대체
    ('encoder', OneHotEncoder(handle_unknown='ignore'))  # 원-핫 인코딩 (drop_first 제거)
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

# 타겟 및 특징 변수 설정
X = data.drop(columns=['survived'], errors='ignore')
y = data['survived'].dropna()

# 결측치 제거 (타겟이 결측치인 경우)
X = X.loc[y.index]


## 모델을 여러 개 추가하자!
log_reg=LogisticRegression(solver='liblinear')
gbm=GradientBoostingClassifier()
xgb=XGBClassifier(use_label_encoder= False, eval_metric ='logloss')

## Voting으로 묶기

voting_clf =VotingClassifier(
    estimators = [
        ('log_reg',log_reg),
        ('gbm',gbm),
        ('xgb',xgb)        
    ], voting='soft')

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', voting_clf)
])

# 교차 검증 수행
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

# 결과 출력
print(f'교차 검증 정확도 평균: {scores.mean():.4f}')
print(f'교차 검증 정확도 표준편차: {scores.std():.4f}')


D:\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:45:51] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
D:\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:45:52] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
D:\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:45:53] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
D:\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:45:54] WAR

교차 검증 정확도 평균: 0.8260
교차 검증 정확도 표준편차: 0.0146
